# How long will this take?

Clustering methods differ in runtime by **orders of magnitude**, and the ranking changes with the
size of your data — so a benchmark on a small sample will mislead you.

In [ ]:
import statistics
import warnings

import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig

pio.renderers.default = "notebook_connected"
warnings.filterwarnings("ignore")  # keep the timing tables clean

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
print(f"{len(raw)} hours = {len(raw) // 24} days")

## Measure it honestly

Two precautions, or the numbers are noise:

* **Warm up first.** The first clustering in a session pays one-off setup — BLAS thread pools,
  solver initialisation — that belongs to no algorithm.
* **Repeat the cheap ones.** Sub-second timings vary run to run, so take a median. Slow ones are
  measured once.

`result.clustering_duration` reports the **clustering step alone**, excluding loading,
normalisation and reconstruction.

In [ ]:
def measure(data, method, n_clusters=8, period_duration="1D"):
    """Clustering time in seconds, plus the result, for one configuration."""

    def once():
        return tsam.aggregate(
            data,
            n_clusters=n_clusters,
            period_duration=period_duration,
            cluster=ClusterConfig(method=method),
        )

    result = once()
    seconds = result.clustering_duration
    if seconds < 0.5:  # cheap enough that noise dominates — take a median
        seconds = statistics.median(
            [seconds, once().clustering_duration, once().clustering_duration]
        )
    return seconds, result

## How runtime grows with the number of periods

Six methods, four dataset sizes, 8 typical days each. This one sweep produces every number on the
page. The warm-up covers **every method** — each backend has its own first-call cost, and warming
only one leaves a several-second phantom on whichever method comes first.

In [ ]:
methods = ["averaging", "hierarchical", "contiguous", "kmeans", "kmaxoids", "kmedoids"]
sizes = [45, 90, 180, 365]

warmup = raw.iloc[: 20 * 24]
for method in methods:  # untimed: absorb each backend's one-off setup
    tsam.aggregate(
        warmup, n_clusters=4, period_duration="1D", cluster=ClusterConfig(method=method)
    )

rows = []
for n_days in sizes:
    data = raw.iloc[: n_days * 24]
    for method in methods:
        seconds, result = measure(data, method)
        rows.append(
            {
                "days": n_days,
                "method": method,
                "seconds": seconds,
                "mean RMSE": float(result.accuracy.rmse.mean()),
            }
        )
scaling = pd.DataFrame(rows)
scaling.pivot(index="days", columns="method", values="seconds").round(3)

In [ ]:
px.line(
    scaling,
    x="days",
    y="seconds",
    color="method",
    markers=True,
    log_y=True,
    title="Clustering time vs. number of periods (k = 8, log scale)",
    labels={"days": "daily periods clustered", "seconds": "clustering time [s]"},
)

Three tiers, orders of magnitude apart — note the log axis:

* **`averaging`, `hierarchical`, `contiguous`** — effectively free, milliseconds throughout.
* **`kmeans`, `kmaxoids`** — a fraction of a second to a few seconds, growing gently.
* **`kmedoids`** — a different world. It solves an exact **MILP** whose binary variables grow with
  the *square* of the period count, so a year of days takes minutes.

**This is why small-sample benchmarks mislead.** Follow `kmeans` and `kmedoids` down the table: at
the smallest size they are neck and neck; by 90 days they differ by an order of magnitude, by a
year by more than two. `kmeans` is nearly flat — it pays a fixed per-call cost (scikit-learn
restarts it ten times) that swamps the real work at every size here. Measure on a month and you
will pick the wrong method for a year.

## What you get for the time

Runtime only matters next to what it buys. At a full year of days:

In [ ]:
(
    scaling[scaling["days"] == 365]
    .set_index("method")[["seconds", "mean RMSE"]]
    .round({"seconds": 3, "mean RMSE": 4})
    .sort_values("seconds")
)

`kmedoids` costs **thousands of times** more than `hierarchical` without buying a better fit — it
edges out `hierarchical` and loses to `kmeans`, which is some two hundred times faster. The time
buys a *property*, not accuracy: centres that are real observed periods, chosen optimally rather
than greedily. See [Clustering methods](clustering_methods.ipynb).

## The lever that matters: period length

Runtime is driven by the **number of periods**, not the amount of data — and `period_duration`
controls that directly. The same 8760 hours, as 52 weekly periods instead of 365 daily ones:

In [ ]:
compared = ["hierarchical", "kmeans", "kmedoids"]
weekly = {m: measure(raw, m, period_duration="1W")[0] for m in compared}
daily = scaling[scaling["days"] == 365].set_index("method")["seconds"]

pd.DataFrame(
    {
        "365 daily periods": {m: round(daily[m], 3) for m in compared},
        "52 weekly periods": {m: round(weekly[m], 3) for m in compared},
        "speed-up": {m: f"{daily[m] / weekly[m]:.0f}x" for m in compared},
    }
)

A **hundredfold speed-up or better** for `kmedoids` on identical data: the MILP shrank from 365²
to 52² binaries. `kmeans` barely moves, because its fixed setup dominates — the lever bites
hardest exactly where runtime hurts.

Two smaller ones: **`n_clusters`** changes MILP difficulty unpredictably (a larger `k` can be
*easier* to prove optimal — measure, don't assume), and **attributes / timesteps per period** widen
each period's vector, affecting distance computation rather than combinatorics.

## If it is too slow

1. **Use the default.** `hierarchical` is milliseconds at any size here and rarely the accuracy
   bottleneck.
2. **Lengthen the period.** `period_duration="1W"` cuts the period count sevenfold.
3. **Swap `kmedoids` for `hierarchical`.** Both give representatives that are real periods; only
   `kmedoids` pays a MILP to prove the choice optimal.
4. **Reduce before clustering.** [Segmentation](segmentation.ipynb) shrinks each period;
   [tuning](tuning.ipynb) searches sizes — but it runs *many* aggregations, so pass `n_jobs` and
   pick a cheap method for the search itself.

---

* [Clustering methods](clustering_methods.ipynb) — what each method guarantees.
* [Comparing clustering methods](../tutorials/comparing_clustering_methods.ipynb) — why they
  disagree.
* [How small can you go?](tuning.ipynb) — the accuracy-vs-size search.